# Baseline models

This notebook evaluates the benchmark forecasting models on the
cleaned train, validation and test splits. Simple statistical models are also fit directly here.

All baseline models return predictions and ground truth in raw value space.
`ForecastEvaluator` is responsible for transforming predictions into the
required evaluation space and computing the common metrics.

The current available evaluation metrics are:

1. **Cumulative log-change MAE**
2. **MASE**
3. **Pearson Correlation between predictions and target in cumulative log change space**
4. **Relative MAE versus Persistence**
5. **Persistence win rate**

Bootstrap evaluation metrics over the test datset are available and used by default.

The available benchmark models are:

1. **Persistence** — predicts every future horizon using the final target
   value in the context window.
2. **Mean** — predicts every future horizon using the mean target value over
   the context window.
3. **ARIMA** — fits a separate univariate ARIMA model to the one-step log
   changes of each asset and target channel.
4. **VAR** — fits one multivariate VAR model per target channel across all
   assets.
5. **GARCH** — fits a separate GARCH(1,1) model to each asset and target
   channel, with an optional AR(1), constant or zero conditional mean.
6. **ModernTCN** - multiple ablations have been trained in Colab. The
    best version checkpoint (based on validation loss) is called and used to 
    predict. It is the version which takes all OHLCV as input, adds a time
    of day temporal feature and flattens series into batch (so we dont mix
    across asset - huge reduction in parameter count - 121,138 params in total
    including 256 params added for the temporal feature).

In [1]:
from pathlib import Path
import sys
from time import perf_counter
import pandas as pd
import torch

# Make sure notebook can import from src/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from src.data.load_candle_data import load_candle_splits, clean_candle_splits
from src.evaluation.metrics import ForecastEvaluator
from src.models.persistence import PersistenceBaseline
from src.models.mean import MeanBaseline
from src.models.arima import ArimaBaseline
from src.models.var import VarBaseline
from src.models.garch import GarchBaseline
from src.models.modern_tcn import ModernTCNBaseline
from src.utils.config import load_yaml
from src.utils.metric_tables import make_evaluation_table

Project root: /Users/vishalruparelia/Desktop/Thesis/dynamic_graphs_thesis


In [2]:
DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "Shared drives/Vishal/data/cached_datasets/"
    "exp-1m-95s-24y/session"
)

CONFIG_PATH = Path("../configs/forecasting.yaml")

## Load the data and clean

In [20]:
config = load_yaml(CONFIG_PATH)

train_raw, val_raw, test_raw = load_candle_splits(DATA_DIR)

train, val, test = clean_candle_splits(
    train_raw,
    val_raw,
    test_raw,
)

print("train samples:", len(train["samples"]))
print("val samples:", len(val["samples"]))
print("test samples:", len(test["samples"]))
print("channels:", test["channels"])
print("assets:", len(test["asset_cols"]))
print("stride:", config['forecasting']['stride'])
print("input features:", config['forecasting']['input_channels'])
print("targets:", config['forecasting']['target_channels'])

#Set global Bootstrap params
BOOTSTRAP_N = 10_000
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95
BOOTSTRAP_SEED = 42


train samples: 167
val samples: 20
test samples: 62
channels: ['open', 'high', 'low', 'close', 'volume', 'amount']
assets: 93
stride: 15
input features: ['open', 'high', 'low', 'close', 'volume']
targets: ['close']


## Persistence

In [ ]:
persistence = PersistenceBaseline.from_config(
    config
)

persistence.fit(
    train_split=train,
    val_split=val,
)

persistence_result = persistence.predict(
    split=test,
    batch_size=256,
)

persistence_evaluator = ForecastEvaluator(
    prediction_result=persistence_result,
    train_split=train,
)


# Persistence predicts zero cumulative log change at every horizon,
# so its cumulative-log-change Pearson correlation is undefined.
persistence_metric_names = [
    metric_name
    for metric_name in persistence_evaluator.available_metrics
    if metric_name
    != "cumulative_log_change_pearson_correlation"
]


persistence_results = persistence_evaluator.evaluate(
    metrics=persistence_metric_names,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

persistence_metric_table = make_evaluation_table(
    metric_results=persistence_results,
    horizons=persistence_evaluator.horizons,
    channels=persistence_evaluator.channels,
)


for metric_name in persistence_metric_names:
    metric_display = (
        persistence_metric_table
        .loc[
            persistence_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000367,0.000352,0.000383,0.000367,0.000008
5,close,0.000785,0.000757,0.000815,0.000785,0.000015
15,close,0.001322,0.001276,0.001371,0.001322,0.000025
30,close,0.001838,0.001767,0.001914,0.001839,0.000038
60,close,0.002553,0.002428,0.002697,0.002554,0.000069


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.949704,0.911210,0.991452,0.949813,0.020469
5,close,2.041615,1.968282,2.119840,2.041683,0.038680
15,close,3.432314,3.312254,3.561779,3.432670,0.063754
30,close,4.775831,4.587843,4.975652,4.776736,0.099277
60,close,6.649479,6.324530,7.021874,6.651386,0.178045


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.000000,1.000000,1.000000,1.000000,0.000000
5,close,1.000000,1.000000,1.000000,1.000000,0.000000
15,close,1.000000,1.000000,1.000000,1.000000,0.000000
30,close,1.000000,1.000000,1.000000,1.000000,0.000000
60,close,1.000000,1.000000,1.000000,1.000000,0.000000


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.500000,0.500000,0.500000,0.500000,0.000000
5,close,0.500000,0.500000,0.500000,0.500000,0.000000
15,close,0.500000,0.500000,0.500000,0.500000,0.000000
30,close,0.500000,0.500000,0.500000,0.500000,0.000000
60,close,0.500000,0.500000,0.500000,0.500000,0.000000


## Mean

In [6]:
mean = MeanBaseline.from_config(
    config
)

mean.fit(
    train_split=train,
    val_split=val,
)

mean_result = mean.predict(
    split=test,
    batch_size=256,
)

mean_evaluator = ForecastEvaluator(
    prediction_result=mean_result,
    train_split=train,
)

mean_results = mean_evaluator.evaluate(
    metrics=mean_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

mean_metric_table = make_evaluation_table(
    metric_results=mean_results,
    horizons=mean_evaluator.horizons,
    channels=mean_evaluator.channels,
)

for metric_name in mean_evaluator.available_metrics:
    metric_display = (
        mean_metric_table
        .loc[
            mean_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.001651,0.001582,0.001725,0.001651,0.000037
5,close,0.001787,0.001711,0.001867,0.001787,0.000040
15,close,0.002075,0.001984,0.002172,0.002075,0.000048
30,close,0.002429,0.002317,0.002550,0.002429,0.000060
60,close,0.003012,0.002851,0.003197,0.003013,0.000089


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.001426,-0.027209,0.032355,0.001558,0.015368
5,close,0.009109,-0.007818,0.027211,0.009177,0.008866
15,close,0.000604,-0.026585,0.026697,0.001032,0.013675
30,close,0.000925,-0.035819,0.035174,0.001521,0.018281
60,close,-0.005952,-0.064780,0.042979,-0.004699,0.028333


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,4.292427,4.111240,4.487903,4.293373,0.096311
5,close,4.646160,4.451149,4.856565,4.647267,0.104283
15,close,5.394403,5.158553,5.651376,5.395818,0.126212
30,close,6.318103,6.029741,6.632837,6.320128,0.156057
60,close,7.850683,7.438304,8.321093,7.853870,0.226875


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,4.477811,4.345123,4.616718,4.479067,0.069203
5,close,2.234747,2.185501,2.286766,2.235230,0.025720
15,close,1.553153,1.514623,1.594874,1.553365,0.020413
30,close,1.314323,1.287402,1.343726,1.314403,0.014433
60,close,1.171088,1.149794,1.192419,1.171135,0.010806


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.136179,0.132218,0.140506,0.136146,0.002087
5,close,0.250657,0.244856,0.256394,0.250589,0.002919
15,close,0.334547,0.327792,0.341339,0.334461,0.003459
30,close,0.377161,0.370388,0.383665,0.377076,0.003318
60,close,0.408607,0.401368,0.415745,0.408506,0.003709


## ARIMA

In [ ]:
arima = ArimaBaseline.from_config(
    config,
    fit_mode="simple",
    optim_method="powell",
)

arima.fit(
    train_split=train,
    val_split=val,
)

arima_result = arima.predict(
    split=test,
    batch_size=32,
)

arima_evaluator = ForecastEvaluator(
    prediction_result=arima_result,
    train_split=train,
)

arima_results = arima_evaluator.evaluate(
    metrics=arima_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

arima_metric_table = make_evaluation_table(
    metric_results=arima_results,
    horizons=arima_evaluator.horizons,
    channels=arima_evaluator.channels,
)

for metric_name in arima_evaluator.available_metrics:
    metric_display = (
        arima_metric_table
        .loc[
            arima_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

Fitting 372 ARIMA models using fit_mode='simple'...
  fitted 25/372
  fitted 50/372
  fitted 75/372
  fitted 100/372
  fitted 125/372
  fitted 150/372
  fitted 175/372
  fitted 200/372
  fitted 225/372
  fitted 250/372
  fitted 275/372
  fitted 300/372
  fitted 325/372
  fitted 350/372
Finished fitting ARIMA models.
Failed models: 0


channel,close,high,low,open
horizon,,,,
1,0.000367,0.000327,0.000327,0.000357
5,0.000785,0.000776,0.000781,0.000797
15,0.001323,0.001315,0.001325,0.001328
30,0.001840,0.001835,0.001847,0.001848
60,0.002558,0.002553,0.002577,0.002568


channel,close,high,low,open
horizon,,,,
1,0.031713,0.097152,0.092515,0.013033
5,0.015962,0.045084,0.040774,0.001215
15,-0.001287,0.029390,0.032286,0.000559
30,0.000689,0.017838,0.020862,0.002072
60,-0.000526,0.009527,0.008055,0.000107


channel,close,high,low,open
horizon,,,,
1,0.951805,0.974656,0.957723,0.945668
5,2.042834,2.309532,2.290382,2.115750
15,3.435128,3.899825,3.877391,3.518914
30,4.781093,5.445918,5.409329,4.899085
60,6.664249,7.594147,7.572843,6.828806


channel,close,high,low,open
horizon,,,,
1,1.004039,0.999840,1.000875,1.000749
5,0.998848,0.999057,0.999103,1.000065
15,1.000198,0.999931,1.000173,1.000370
30,1.000399,1.000285,1.001035,1.000371
60,1.000953,1.000660,1.003050,1.001041


channel,close,high,low,open
horizon,,,,
1,0.458436,0.477522,0.474579,0.457172
5,0.486272,0.497736,0.492730,0.482214
15,0.489649,0.497777,0.491018,0.487499
30,0.489731,0.495340,0.487061,0.491265
60,0.485103,0.497326,0.479923,0.485012


## VAR

In [7]:
var = VarBaseline.from_config(
    config,
    maxlags=15,
    ic="aic",
    trend="c",
)

var.fit(
    train_split=train,
    val_split=val,
)

var_result = var.predict(
    split=test,
    batch_size=256,
)

var_evaluator = ForecastEvaluator(
    prediction_result=var_result,
    train_split=train,
)

var_results = var_evaluator.evaluate(
    metrics=var_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

var_metric_table = make_evaluation_table(
    metric_results=var_results,
    horizons=var_evaluator.horizons,
    channels=var_evaluator.channels,
)

for metric_name in var_evaluator.available_metrics:
    metric_display = (
        var_metric_table
        .loc[
            var_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

Fitting 1 VAR model(s) with maxlags=15, ic=aic...
  close: selected_lag=11, failed=False
Finished fitting VAR models.
Failed models: 0


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000379,0.000364,0.000395,0.000379,0.000008
5,close,0.000799,0.000770,0.000830,0.000799,0.000015
15,close,0.001335,0.001288,0.001385,0.001335,0.000025
30,close,0.001848,0.001776,0.001924,0.001848,0.000038
60,close,0.002563,0.002437,0.002707,0.002563,0.000070


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.010872,-0.004869,0.026517,0.010967,0.008019
5,close,0.015784,0.000930,0.031262,0.015887,0.007697
15,close,-0.002537,-0.018208,0.012744,-0.002487,0.007854
30,close,0.004379,-0.012439,0.020928,0.004453,0.008567
60,close,0.002263,-0.012588,0.017160,0.002322,0.007576


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.982359,0.944070,1.024233,0.982443,0.020322
5,close,2.076806,2.002656,2.156407,2.076869,0.039204
15,close,3.464684,3.342842,3.597083,3.465064,0.064807
30,close,4.800755,4.611415,5.002028,4.801647,0.100122
60,close,6.675028,6.347023,7.049401,6.676942,0.179296


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.039215,1.034092,1.044508,1.039190,0.002680
5,close,1.019413,1.014942,1.023997,1.019414,0.002308
15,close,1.008641,1.005642,1.011762,1.008662,0.001555
30,close,1.004279,1.001789,1.006703,1.004283,0.001254
60,close,1.003097,1.001223,1.005025,1.003100,0.000971


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.434530,0.429501,0.439546,0.434573,0.002584
5,close,0.469837,0.464981,0.474520,0.469835,0.002449
15,close,0.475889,0.470768,0.480904,0.475890,0.002577
30,close,0.482826,0.477335,0.488307,0.482834,0.002784
60,close,0.484953,0.477632,0.492155,0.484942,0.003711


## GARCH

In [8]:
garch = GarchBaseline.from_config(
    config,
    mean="AR",
    return_scale=10000.0,
)

garch.fit(
    train_split=train,
    val_split=val,
)

garch_result = garch.predict(
    split=test,
    batch_size=256,
)

garch_evaluator = ForecastEvaluator(
    prediction_result=garch_result,
    train_split=train,
)

garch_results = garch_evaluator.evaluate(
    metrics=garch_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

garch_metric_table = make_evaluation_table(
    metric_results=garch_results,
    horizons=garch_evaluator.horizons,
    channels=garch_evaluator.channels,
)

for metric_name in garch_evaluator.available_metrics:
    metric_display = (
        garch_metric_table
        .loc[
            garch_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

Fitting 93 GARCH(1,1) models with mean='AR'...
  fitted 25/93
  fitted 50/93
  fitted 75/93
Finished fitting GARCH models.
Failed models: 0


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000368,0.000353,0.000384,0.000368,0.000008
5,close,0.000785,0.000757,0.000816,0.000785,0.000015
15,close,0.001323,0.001277,0.001373,0.001323,0.000025
30,close,0.001841,0.001769,0.001917,0.001841,0.000038
60,close,0.002560,0.002433,0.002706,0.002561,0.000070


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.031935,0.005737,0.055063,0.031912,0.012540
5,close,0.014379,-0.002126,0.031008,0.014254,0.008470
15,close,0.000941,-0.015292,0.015555,0.000997,0.007813
30,close,0.002778,-0.010144,0.015023,0.002865,0.006420
60,close,0.003407,-0.010374,0.017557,0.003569,0.007142


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.952390,0.913917,0.994147,0.952497,0.020432
5,close,2.042920,1.969624,2.121329,2.042993,0.038719
15,close,3.435790,3.315743,3.565393,3.436143,0.063774
30,close,4.782484,4.594722,4.982995,4.783394,0.099694
60,close,6.668193,6.338200,7.044449,6.670116,0.180179


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.005828,1.003290,1.008365,1.005813,0.001296
5,close,0.999117,0.998180,0.999993,0.999119,0.000466
15,close,1.000455,0.999625,1.001283,1.000452,0.000427
30,close,1.000854,0.999480,1.002257,1.000850,0.000706
60,close,1.001783,0.999338,1.004198,1.001772,0.001233


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.456903,0.451275,0.462653,0.456904,0.002926
5,close,0.485596,0.478777,0.492378,0.485561,0.003449
15,close,0.487458,0.478490,0.496458,0.487474,0.004594
30,close,0.487705,0.475254,0.500224,0.487707,0.006411
60,close,0.482064,0.464182,0.499740,0.482045,0.009051


In [21]:
modern_tcn_checkpoint_path = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation/checkpoints/modern_tcn/"
    "per_asset_ohlcv_to_c_tod/runs/xff03ri9/"
    "best_checkpoint.pt"
).expanduser().resolve()

modern_tcn = ModernTCNBaseline.from_config(
    config
)

modern_tcn.load_checkpoint(
    checkpoint_path=modern_tcn_checkpoint_path,
    device="cpu",
)

modern_tcn_result = modern_tcn.predict(
    split=test,
    batch_size=8,
    num_workers=0,
)

modern_tcn_evaluator = ForecastEvaluator(
    prediction_result=modern_tcn_result,
    train_split=train,
)

modern_tcn_results = modern_tcn_evaluator.evaluate(
    metrics=modern_tcn_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

modern_tcn_metric_table = make_evaluation_table(
    metric_results=modern_tcn_results,
    horizons=modern_tcn_evaluator.horizons,
    channels=modern_tcn_evaluator.channels,
)

for metric_name in modern_tcn_evaluator.available_metrics:
    metric_display = (
        modern_tcn_metric_table
        .loc[
            modern_tcn_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style.set_caption(
            metric_name
        )
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000375,0.000361,0.000391,0.000375,0.000008
5,close,0.000789,0.000760,0.000819,0.000789,0.000015
15,close,0.001326,0.001280,0.001376,0.001326,0.000025
30,close,0.001842,0.001770,0.001919,0.001843,0.000038
60,close,0.002563,0.002435,0.002711,0.002564,0.000071


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.019029,0.002766,0.035888,0.019081,0.008434
5,close,0.010166,-0.007365,0.027989,0.010095,0.009048
15,close,0.001344,-0.021419,0.020667,0.001398,0.010782
30,close,-0.010835,-0.040338,0.016895,-0.010715,0.014779
60,close,-0.027869,-0.079314,0.014708,-0.027062,0.024469


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.973627,0.935685,1.014966,0.973718,0.020248
5,close,2.052016,1.978222,2.131113,2.052102,0.039070
15,close,3.443669,3.323600,3.573125,3.443989,0.063909
30,close,4.787367,4.597632,4.988654,4.788282,0.100085
60,close,6.676185,6.343196,7.058268,6.678105,0.181992


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.030338,1.025803,1.034843,1.030317,0.002312
5,close,1.004047,1.001681,1.006409,1.004046,0.001221
15,close,1.003616,1.001640,1.005577,1.003601,0.001005
30,close,1.002709,1.001153,1.004339,1.002711,0.000812
60,close,1.003994,1.001622,1.006579,1.003975,0.001263


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.438382,0.432079,0.444685,0.438419,0.003226
5,close,0.477518,0.472365,0.482703,0.477486,0.002634
15,close,0.486765,0.480790,0.492744,0.486812,0.003034
30,close,0.485710,0.477130,0.493939,0.485711,0.004255
60,close,0.481799,0.470635,0.492442,0.481778,0.005561
